# Seller Performance — Анализ продавцов

## Цель анализа

Цель этапа — оценить вклад продавцов в товарный оборот платформы и определить продавцов, требующих внимания с точки зрения клиентского опыта.

В рамках анализа:
- рассчитываются ключевые показатели продавцов;
- оценивается концентрация товарного оборота;
- сравниваются рейтинги продавцов с достаточным числом отзывов;
- исследуется связь между сроком доставки и оценкой;
- формируется список приоритетных продавцов для дальнейшего анализа.

## 1. Подготовка данных

Для анализа используются таблицы:

- `order_items` — товары, стоимость, продавец и заказ;
- `orders` — статус заказа и даты доставки;
- `reviews` — оценки клиентов;
- `sellers` — регион продавца.

Агрегация сначала выполняется на уровне пары `seller_id`–`order_id`. Это предотвращает дублирование оценки заказа, если один продавец продал в заказе несколько товаров.

В анализ включаются только доставленные заказы. Если заказ содержит товары нескольких продавцов, его оценка относится ко всем участвующим продавцам; это ограничение необходимо учитывать при интерпретации.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from scipy.stats import spearmanr

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

In [ ]:
DATA_PATH = Path("../data")

orders = pd.read_csv(
    DATA_PATH / "olist_orders_dataset.csv"
)

items = pd.read_csv(
    DATA_PATH / "olist_order_items_dataset.csv"
)

reviews = pd.read_csv(
    DATA_PATH / "olist_order_reviews_dataset.csv"
)

sellers = pd.read_csv(
    DATA_PATH / "olist_sellers_dataset.csv"
)

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"],
    errors="coerce"
)

reviews_one_per_order = (
    reviews
    .sort_values(
        ["order_id", "review_answer_timestamp"],
        na_position="first"
    )
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
)

delivered_orders = (
    orders
    .loc[orders["order_status"].eq("delivered")]
    .copy()
)

delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds().div(86_400)

delivered_orders["delay_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_estimated_delivery_date"]
).dt.total_seconds().div(86_400)

delivered_orders["is_late"] = (
    delivered_orders["delay_days"]
    .gt(0)
    .where(delivered_orders["delay_days"].notna())
)

seller_orders = (
    items
    .groupby(
        ["seller_id", "order_id"],
        as_index=False
    )
    .agg(
        order_product_value=("price", "sum"),
        order_freight_value=("freight_value", "sum"),
        items_count=("order_item_id", "count")
    )
    .merge(
        delivered_orders[
            [
                "order_id",
                "delivery_days",
                "delay_days",
                "is_late"
            ]
        ],
        on="order_id",
        how="inner",
        validate="many_to_one"
    )
    .merge(
        reviews_one_per_order[
            ["order_id", "review_score"]
        ],
        on="order_id",
        how="left",
        validate="many_to_one"
    )
)

seller_orders = (
    seller_orders
    .loc[
        seller_orders["delivery_days"].isna()
        | seller_orders["delivery_days"].ge(0)
    ]
    .copy()
)

seller_stats = (
    seller_orders
    .groupby(
        "seller_id",
        as_index=False
    )
    .agg(
        orders=("order_id", "nunique"),
        product_gmv=("order_product_value", "sum"),
        avg_order_value=("order_product_value", "mean"),
        items_sold=("items_count", "sum"),
        reviews_count=("review_score", "count"),
        avg_review=("review_score", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        late_share=("is_late", "mean")
    )
    .merge(
        sellers[
            ["seller_id", "seller_state"]
        ].drop_duplicates("seller_id"),
        on="seller_id",
        how="left",
        validate="one_to_one"
    )
)

seller_stats = seller_stats.sort_values(
    "product_gmv",
    ascending=False
)

seller_stats.head()

Показатель `product_gmv` рассчитан как сумма стоимости товаров (`price`) и не включает стоимость доставки. Это товарный оборот через продавца, а не чистая выручка или прибыль продавца.

## 2. Ключевые показатели продавцов

Рассчитываются:
- количество активных продавцов;
- объём товарного оборота;
- число заказов;
- средняя стоимость заказа на продавца;
- средняя оценка;
- среднее время доставки;
- доля опоздавших заказов.

In [ ]:
seller_kpis = pd.DataFrame(
    {
        "Метрика": [
            "Активные продавцы",
            "Доставленные seller-order",
            "Товарный оборот, BRL",
            "Медианное число заказов на продавца",
            "Средняя оценка по отзывам",
            "Средняя доля опозданий"
        ],
        "Значение": [
            f"{seller_stats['seller_id'].nunique():,}".replace(",", " "),
            f"{len(seller_orders):,}".replace(",", " "),
            f"{seller_stats['product_gmv'].sum():,.0f}".replace(",", " "),
            f"{seller_stats['orders'].median():.0f}",
            f"{seller_orders['review_score'].mean():.2f}",
            f"{seller_orders['is_late'].mean():.1%}"
        ]
    }
)

seller_kpis

## 3. Продавцы с наибольшим товарным оборотом

Топ продавцов определяется по сумме стоимости товаров в доставленных заказах.

In [ ]:
top_sellers = (
    seller_stats
    .head(10)
    .sort_values(
        "product_gmv",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.barh(
    top_sellers["seller_id"],
    top_sellers["product_gmv"]
)

ax.set_title(
    "Топ-10 продавцов по товарному обороту"
)
ax.set_xlabel(
    "Товарный оборот, BRL"
)
ax.set_ylabel(
    "ID продавца"
)

for bar, value in zip(
    bars,
    top_sellers["product_gmv"]
):
    ax.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f" {value:,.0f}",
        va="center"
    )

plt.tight_layout()
plt.show()

## 4. Концентрация товарного оборота

Для оценки зависимости платформы от крупнейших продавцов рассчитываются:
- доля топ-1;
- доля топ-5;
- доля топ-10;
- индекс HHI.

HHI используется как описательная мера концентрации: чем выше показатель, тем сильнее оборот сосредоточен у ограниченного числа продавцов.

In [ ]:
total_gmv = seller_stats["product_gmv"].sum()

seller_stats["gmv_share"] = (
    seller_stats["product_gmv"]
    / total_gmv
)

concentration = pd.DataFrame(
    {
        "Показатель": [
            "Доля топ-1",
            "Доля топ-5",
            "Доля топ-10",
            "HHI"
        ],
        "Значение": [
            seller_stats.head(1)["gmv_share"].sum(),
            seller_stats.head(5)["gmv_share"].sum(),
            seller_stats.head(10)["gmv_share"].sum(),
            (
                seller_stats["gmv_share"]
                .mul(100)
                .pow(2)
                .sum()
            )
        ]
    }
)

concentration_display = concentration.copy()
concentration_display["Значение"] = (
    concentration_display["Значение"].astype(object)
)

share_mask = concentration_display["Показатель"].ne("HHI")

concentration_display.loc[
    share_mask,
    "Значение"
] = concentration_display.loc[
    share_mask,
    "Значение"
].map(lambda value: f"{value:.1%}")

concentration_display.loc[
    ~share_mask,
    "Значение"
] = concentration_display.loc[
    ~share_mask,
    "Значение"
].map(lambda value: f"{value:.1f}")

concentration_display

## 5. Сравнение качества работы продавцов

Средняя оценка нестабильна у продавцов с небольшим числом отзывов. Поэтому:
- для сравнения используются продавцы минимум с 30 отзывами;
- дополнительно рассчитывается взвешенный рейтинг, который сглаживает случайно высокие или низкие оценки при небольшой выборке.

Формула учитывает среднюю оценку продавца, число его отзывов и среднюю оценку по платформе.

In [ ]:
MIN_REVIEWS = 30

global_review_mean = (
    seller_orders["review_score"].mean()
)

eligible_sellers = (
    seller_stats
    .loc[
        seller_stats["reviews_count"].ge(MIN_REVIEWS)
    ]
    .copy()
)

eligible_sellers["weighted_rating"] = (
    (
        eligible_sellers["reviews_count"]
        / (
            eligible_sellers["reviews_count"]
            + MIN_REVIEWS
        )
    )
    * eligible_sellers["avg_review"]
    +
    (
        MIN_REVIEWS
        / (
            eligible_sellers["reviews_count"]
            + MIN_REVIEWS
        )
    )
    * global_review_mean
)

best_sellers = (
    eligible_sellers
    .nlargest(
        10,
        "weighted_rating"
    )[
        [
            "seller_id",
            "orders",
            "product_gmv",
            "reviews_count",
            "avg_review",
            "weighted_rating",
            "avg_delivery_days",
            "late_share"
        ]
    ]
)

worst_sellers = (
    eligible_sellers
    .nsmallest(
        10,
        "weighted_rating"
    )[
        [
            "seller_id",
            "orders",
            "product_gmv",
            "reviews_count",
            "avg_review",
            "weighted_rating",
            "avg_delivery_days",
            "late_share"
        ]
    ]
)

display(
    Markdown("### Продавцы с наиболее высоким взвешенным рейтингом")
)
display(best_sellers)

display(
    Markdown("### Продавцы с наиболее низким взвешенным рейтингом")
)
display(worst_sellers)

## 6. Связь доставки и оценки продавца

Для продавцов минимум с 30 отзывами рассчитывается корреляция Спирмена между:
- средним временем доставки;
- средней оценкой клиента.

Корреляция показывает направление и силу монотонной связи, но не доказывает причинность.

In [ ]:
correlation_data = (
    eligible_sellers
    .dropna(
        subset=[
            "avg_delivery_days",
            "avg_review"
        ]
    )
)

spearman_corr, spearman_p = spearmanr(
    correlation_data["avg_delivery_days"],
    correlation_data["avg_review"]
)

correlation_result = pd.DataFrame(
    {
        "Показатель": [
            "Корреляция Спирмена",
            "p-value",
            "Количество продавцов"
        ],
        "Значение": [
            spearman_corr,
            spearman_p,
            len(correlation_data)
        ]
    }
)

correlation_result

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    correlation_data["avg_delivery_days"],
    correlation_data["avg_review"],
    alpha=0.6
)

if len(correlation_data) >= 2:
    slope, intercept = np.polyfit(
        correlation_data["avg_delivery_days"],
        correlation_data["avg_review"],
        1
    )

    trend_x = np.linspace(
        correlation_data["avg_delivery_days"].min(),
        correlation_data["avg_delivery_days"].max(),
        100
    )

    ax.plot(
        trend_x,
        slope * trend_x + intercept,
        linestyle="--",
        label="Линейный тренд"
    )

    ax.legend()

ax.set_title(
    "Средняя оценка и время доставки по продавцам"
)
ax.set_xlabel(
    "Среднее время доставки, дней"
)
ax.set_ylabel(
    "Средняя оценка"
)
ax.set_ylim(1, 5)

plt.tight_layout()
plt.show()

## 7. Приоритетные продавцы для детального анализа

Для практического применения выделяются продавцы, которые:
- входят в верхний квартиль по товарному обороту;
- имеют оценку ниже медианы или долю опозданий выше медианы среди продавцов с достаточным числом отзывов.

Такой сегмент сочетает заметный вклад в бизнес с потенциальной проблемой качества.

In [ ]:
gmv_threshold = seller_stats["product_gmv"].quantile(0.75)
review_threshold = eligible_sellers["avg_review"].median()
late_share_threshold = eligible_sellers["late_share"].median()

priority_sellers = (
    eligible_sellers
    .loc[
        eligible_sellers["product_gmv"].ge(gmv_threshold)
        & (
            eligible_sellers["avg_review"].lt(review_threshold)
            | eligible_sellers["late_share"].gt(late_share_threshold)
        )
    ]
    .sort_values(
        "product_gmv",
        ascending=False
    )[
        [
            "seller_id",
            "seller_state",
            "orders",
            "product_gmv",
            "avg_review",
            "weighted_rating",
            "avg_delivery_days",
            "late_share"
        ]
    ]
)

priority_sellers.head(15)

## 8. Business Insights

In [ ]:
top_1_share = seller_stats.head(1)["gmv_share"].sum()
top_10_share = seller_stats.head(10)["gmv_share"].sum()

corr_description = (
    "отрицательная"
    if spearman_corr < 0
    else "положительная"
)

significance_text = (
    "статистически значима"
    if spearman_p < 0.05
    else "не является статистически значимой"
)

insights = f"""
Основные выводы:

1. В доставленных заказах представлены **{seller_stats['seller_id'].nunique():,} продавцов**. Общий товарный оборот по стоимости товаров составляет **{total_gmv:,.0f} BRL**.

2. Крупнейший продавец формирует **{top_1_share:.1%}** оборота, а топ-10 — **{top_10_share:.1%}**. Следовательно, оборот не сосредоточен только у нескольких крупнейших продавцов.

3. Для корректного сравнения качества отобраны **{len(eligible_sellers):,} продавцов** минимум с {MIN_REVIEWS} отзывами. Взвешенный рейтинг снижает влияние случайных экстремальных оценок при небольшой выборке.

4. Корреляция Спирмена между средним временем доставки и средней оценкой равна **{spearman_corr:.3f}**: связь {corr_description} и {significance_text} (`p-value = {spearman_p:.3g}`).

5. В сегмент для детального анализа попали **{len(priority_sellers):,} продавцов** с высоким оборотом и потенциальными проблемами качества или сроков доставки.

### Рекомендации

- использовать совместно товарный оборот, взвешенный рейтинг и долю опозданий, а не оценивать продавца по одной метрике;
- приоритизировать работу с продавцами из таблицы `priority_sellers`;
- отдельно исследовать причины опозданий: регион, перевозчик, категория товара и этап передачи заказа;
- учитывать, что отзыв выставляется заказу целиком и в заказах с несколькими продавцами не позволяет точно определить вклад каждого продавца.
"""

display(Markdown(insights))